# Multi-provider LLM and embeddings demo

This notebook uses LangChain for Gemini, Groq, OpenAI GPT, and Anthropic Claude chat models, plus Gemini and OpenAI embeddings. LangGraph runs with the chat provider selected in `ai/.env`.

## Setup

1. Run the package-install cell, then restart the notebook kernel if prompted.
2. Configure API keys, provider names, and model IDs in `ai/.env`. This notebook loads them directly and does not print secret values.
3. Run the demo cells you need. Each model demo sends a real API request and may use quota or incur charges.

Set `LLM_PROVIDER` to `gemini`, `groq`, `openai`, or `anthropic`; set `EMBEDDING_PROVIDER` to `gemini` or `openai`.


In [ ]:
%pip install -q langchain-core langgraph langchain-google-genai langchain-groq langchain-openai langchain-anthropic python-dotenv numpy


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# Resolve the AI .env whether the notebook starts from the workspace or ai/ directory.
NOTEBOOK_DIR = Path.cwd()
AI_ENV_FILE = next(
    (candidate for candidate in (
        NOTEBOOK_DIR / "zomato-ai-data-engineering-end-to-end-project" / "ai" / ".env",
        NOTEBOOK_DIR / "ai" / ".env",
        NOTEBOOK_DIR / ".env",
    ) if candidate.is_file()),
    None,
)
if AI_ENV_FILE is None:
    raise FileNotFoundError("Could not find ai/.env. Start Jupyter from the workspace or ai directory.")
load_dotenv(AI_ENV_FILE, override=False)

CHAT_PROVIDERS = {
    "gemini": ("GEMINI_API_KEY", "GEMINI_CHAT_MODEL"),
    "groq": ("GROQ_API_KEY", "GROQ_CHAT_MODEL"),
    "openai": ("OPENAI_API_KEY", "OPENAI_CHAT_MODEL"),
    "anthropic": ("ANTHROPIC_API_KEY", "ANTHROPIC_CHAT_MODEL"),
}
EMBEDDING_PROVIDERS = {
    "gemini": ("GEMINI_API_KEY", "GEMINI_EMBEDDING_MODEL"),
    "openai": ("OPENAI_API_KEY", "OPENAI_EMBEDDING_MODEL"),
}

# LLM_PROVIDER and EMBEDDING_PROVIDER in .env select the demos to run.
CHAT_PROVIDERS_TO_DEMO = [os.getenv("LLM_PROVIDER", "").strip().lower()]
EMBEDDING_PROVIDERS_TO_DEMO = [os.getenv("EMBEDDING_PROVIDER", "").strip().lower()]
GRAPH_PROVIDER = os.getenv("LLM_PROVIDER", "").strip().lower()

for provider in CHAT_PROVIDERS_TO_DEMO:
    if provider not in CHAT_PROVIDERS:
        raise ValueError("Set LLM_PROVIDER in ai/.env to gemini, groq, openai, or anthropic.")
for provider in EMBEDDING_PROVIDERS_TO_DEMO:
    if provider not in EMBEDDING_PROVIDERS:
        raise ValueError("Set EMBEDDING_PROVIDER in ai/.env to gemini or openai.")

print(f"Loaded configuration from {AI_ENV_FILE}")
print("API keys loaded:", sorted({key for key, _ in (*CHAT_PROVIDERS.values(), *EMBEDDING_PROVIDERS.values()) if os.getenv(key)}))


In [ ]:
def get_llm_response(provider, model, prompt, temperature=0):
    """Send a prompt to Gemini, Groq, OpenAI, or Anthropic using credentials from ai/.env.

    Args:
        provider: Provider name: "gemini", "groq", "openai", or "anthropic".
        model: Provider's model ID, or None to use the model ID in ai/.env.
        prompt: User prompt text.
        temperature: Sampling temperature (default 0).

    Returns:
        The model's response text.
    """
    provider = provider.strip().lower()
    if provider not in CHAT_PROVIDERS:
        raise ValueError(f"Unknown provider {provider!r}; choose from {list(CHAT_PROVIDERS)}")

    key_env, model_env = CHAT_PROVIDERS[provider]
    api_key = os.getenv(key_env)
    model_name = model or os.getenv(model_env)
    if not api_key:
        raise ValueError(f"Set {key_env} in {AI_ENV_FILE}")
    if not model_name:
        raise ValueError(f"Pass a model name or set {model_env} in {AI_ENV_FILE}")

    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        llm = ChatGoogleGenerativeAI(model=model_name, google_api_key=api_key, temperature=temperature)
    elif provider == "groq":
        from langchain_groq import ChatGroq
        llm = ChatGroq(model=model_name, api_key=api_key, temperature=temperature)
    elif provider == "openai":
        from langchain_openai import ChatOpenAI
        llm = ChatOpenAI(model=model_name, api_key=api_key, temperature=temperature)
    else:
        from langchain_anthropic import ChatAnthropic
        llm = ChatAnthropic(model=model_name, api_key=api_key, temperature=temperature)

    response = llm.invoke(prompt)
    content = response.content
    if isinstance(content, str):
        return content
    # Some providers return structured content blocks instead of plain text.
    return "\n".join(
        block["text"] if isinstance(block, dict) and "text" in block else str(block)
        for block in content
    )


def build_chat_model(provider, temperature=0):
    """Create a LangChain chat client using credentials and model IDs from ai/.env."""
    if provider not in CHAT_PROVIDERS:
        raise ValueError(f"Unknown chat provider {provider!r}; choose from {list(CHAT_PROVIDERS)}")
    key_env, model_env = CHAT_PROVIDERS[provider]
    api_key = os.getenv(key_env)
    model_name = os.getenv(model_env)
    if not api_key:
        raise ValueError(f"Set {key_env} in {AI_ENV_FILE}")
    if not model_name:
        raise ValueError(f"Set {model_env} in {AI_ENV_FILE}")

    if provider == "gemini":
        from langchain_google_genai import ChatGoogleGenerativeAI
        return ChatGoogleGenerativeAI(model=model_name, google_api_key=api_key, temperature=temperature)
    if provider == "groq":
        from langchain_groq import ChatGroq
        return ChatGroq(model=model_name, api_key=api_key, temperature=temperature)
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=model_name, api_key=api_key, temperature=temperature)
    from langchain_anthropic import ChatAnthropic
    return ChatAnthropic(model=model_name, api_key=api_key, temperature=temperature)


def build_embedding_model(provider):
    """Create a Gemini or OpenAI embedding client using ai/.env values."""
    if provider not in EMBEDDING_PROVIDERS:
        raise ValueError(f"Unknown embedding provider {provider!r}; choose from {list(EMBEDDING_PROVIDERS)}")
    key_env, model_env = EMBEDDING_PROVIDERS[provider]
    api_key = os.getenv(key_env)
    model_name = os.getenv(model_env)
    if not api_key:
        raise ValueError(f"Set {key_env} in {AI_ENV_FILE}")
    if not model_name:
        raise ValueError(f"Set {model_env} in {AI_ENV_FILE}")

    if provider == "gemini":
        from langchain_google_genai import GoogleGenerativeAIEmbeddings
        return GoogleGenerativeAIEmbeddings(model=model_name, google_api_key=api_key)
    from langchain_openai import OpenAIEmbeddings
    return OpenAIEmbeddings(model=model_name, api_key=api_key)

print("Chat models:", {name: os.getenv(config[1]) for name, config in CHAT_PROVIDERS.items()})
print("Embedding models:", {name: os.getenv(config[1]) for name, config in EMBEDDING_PROVIDERS.items()})


## Demo 1 - reusable chat function

Call `get_llm_response(provider, model, prompt)` with a provider name, a model ID (or `None` to use the `.env` model), and prompt text. Provider API keys are read from `ai/.env`.


In [ ]:
# Pass provider, model, and prompt. Use model=None to read that provider's model from ai/.env.
provider = os.getenv("LLM_PROVIDER", "").strip().lower()
model = os.getenv(CHAT_PROVIDERS[provider][1]) if provider in CHAT_PROVIDERS else None
prompt = "Reply with the provider name and one short greeting."

response = get_llm_response(provider=provider, model=model, prompt=prompt)
print(response)


## Demo 2 - embedding model selected in `.env`

Uses `EMBEDDING_PROVIDER`, its API key, and its model ID from `ai/.env`. This cell sends one small batch and one query.


In [ ]:
import numpy as np

sample_documents = [
    "The food arrived hot and the delivery was quick.",
    "The order was late and the meal was cold.",
]
sample_query = "Was the delivery fast?"
embedding_results = {}

for provider in EMBEDDING_PROVIDERS_TO_DEMO:
    if not os.getenv(EMBEDDING_PROVIDERS[provider][0]):
        print(f"Skipping {provider} embeddings: set {EMBEDDING_PROVIDERS[provider][0]} to run this demo.")
        continue
    embeddings = build_embedding_model(provider)
    document_vectors = embeddings.embed_documents(sample_documents)
    query_vector = embeddings.embed_query(sample_query)
    similarities = [
        float(np.dot(query_vector, vector) / (np.linalg.norm(query_vector) * np.linalg.norm(vector)))
        for vector in document_vectors
    ]
    embedding_results[provider] = {
        "model": os.getenv(EMBEDDING_PROVIDERS[provider][1]),
        "dimension": len(query_vector),
        "similarities": similarities,
    }
    print(f"{provider}: dimension={len(query_vector)}, cosine similarities={similarities}")


## Demo 3 - LangGraph with the chat provider selected in `.env`

Reuses `LLM_PROVIDER` and its model configuration. This cell sends a real model request.


In [ ]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph

class DemoState(TypedDict, total=False):
    question: str
    answer: str


def build_chat_graph(provider):
    model = build_chat_model(provider, temperature=0)

    def answer_question(state: DemoState):
        response = model.invoke([
            SystemMessage(content="Answer briefly. This is a LangGraph provider demo."),
            HumanMessage(content=state["question"]),
        ])
        content = response.content
        if not isinstance(content, str):
            content = str(content)
        return {"answer": content}

    graph = StateGraph(DemoState)
    graph.add_node("answer_question", answer_question)
    graph.add_edge(START, "answer_question")
    graph.add_edge("answer_question", END)
    return graph.compile()


demo_graph = build_chat_graph(GRAPH_PROVIDER)
graph_result = demo_graph.invoke({"question": "In one sentence, what is an embedding?"})
print(f"[{GRAPH_PROVIDER}] {graph_result['answer']}")


## Environment variables used

| Purpose | Variables in `ai/.env` |
|---|---|
| Chat provider and model | `LLM_PROVIDER`, plus that provider's `*_API_KEY` and `*_CHAT_MODEL` |
| Embedding provider and model | `EMBEDDING_PROVIDER`, plus its `*_API_KEY` and `*_EMBEDDING_MODEL` |
| Snowflake connection (available to other project code) | `SNOWFLAKE_ACCOUNT`, `SNOWFLAKE_USER`, `SNOWFLAKE_PASSWORD`, `SNOWFLAKE_WAREHOUSE`, `SNOWFLAKE_DATABASE`, `SNOWFLAKE_SCHEMA` |

The notebook reads provider credentials and model IDs from `ai/.env`. It never includes API keys in notebook source or output.


In [ ]:
# new function add here
